# 📊 04 — Model Evaluation
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives
1. Run **quantitative evaluation** on the held-out test set.
2. Compute **Precision, Recall, mAP50, mAP50-95** globally and per class.
3. Visualise **Confusion Matrix**, **PR Curve**, and **F1 Curve**.
4. Identify **weak classes** and understand failure modes.
5. Assess readiness for deployment.

---

### Metrics Glossary

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| **Precision** | TP / (TP + FP) | Of all predictions, how many are correct? |
| **Recall** | TP / (TP + FN) | Of all true objects, how many were found? |
| **F1** | 2·P·R / (P+R) | Harmonic mean — balance of P and R |
| **mAP50** | mean AP @ IoU=0.50 | Standard detection accuracy |
| **mAP50-95** | mean AP @ IoU=0.50→0.95 | Stricter — penalises imprecise boxes |

> **IoU (Intersection-over-Union):** fraction of overlap between predicted
> and ground-truth boxes. Higher IoU = more precise localisation.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Resolve project root by walking up until 'src/' is found
_cwd = Path().resolve()
PROJECT_ROOT = _cwd
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import cv2, random
from ultralytics import YOLO

from src.ppe_detection.utils import (
    MODELS_DIR, CONFIGS_DIR, OUTPUTS_DIR, CLASS_NAMES, PERSON_CLASS_IDS, ensure_dirs
)
from src.ppe_detection.inference import load_model, predict_image, draw_detections

ensure_dirs()
sns.set_theme(style="whitegrid")

WEIGHTS   = MODELS_DIR / "yolov8n_smartmine_baseline.pt"
DATA_YAML = CONFIGS_DIR / "smartmine_unified.yaml"

print(f"Weights : {WEIGHTS}")
print(f"Exists  : {WEIGHTS.exists()}")

## 2. Load Model & Run Test Evaluation

In [ ]:
if not WEIGHTS.exists():
    print("Model not found — run notebook 03 first.")
else:
    model   = YOLO(str(WEIGHTS))
    metrics = model.val(
        data    = str(DATA_YAML),
        split   = "test",
        verbose = False,
    )
    print("Evaluation complete.")

## 3. Overall Metrics

In [ ]:
if WEIGHTS.exists():
    overall = {
        "Precision (mean)"  : round(float(metrics.box.mp),    4),
        "Recall (mean)"     : round(float(metrics.box.mr),    4),
        "mAP50"             : round(float(metrics.box.map50), 4),
        "mAP50-95"          : round(float(metrics.box.map),   4),
    }
    df_overall = pd.DataFrame([overall]).T
    df_overall.columns = ["Score"]

    print("=" * 40)
    print("  TEST SET — OVERALL METRICS")
    print("=" * 40)
    print(df_overall.to_string())
    print("=" * 40)

    # Traffic-light assessment
    map50 = overall["mAP50"]
    if   map50 >= 0.80: grade = "🟢 EXCELLENT"
    elif map50 >= 0.70: grade = "🟡 GOOD"
    elif map50 >= 0.55: grade = "🟠 ACCEPTABLE"
    else:               grade = "🔴 NEEDS IMPROVEMENT"
    print(f"\n  Assessment: {grade} (mAP50={map50})")

## 4. Per-Class Metrics

In [ ]:
if WEIGHTS.exists():
    # metrics.box.p/r/ap50/ap arrays only contain classes that appear in the
    # test set. metrics.box.ap_class_index maps array position -> real class ID.
    ap_class_index = metrics.box.ap_class_index
    class_rows = []
    for arr_idx, class_id in enumerate(ap_class_index):
        cid = int(class_id)
        p   = float(metrics.box.p[arr_idx])
        r   = float(metrics.box.r[arr_idx])
        class_rows.append({
            "id"        : cid,
            "class"     : metrics.names[cid],
            "precision" : round(p, 4),
            "recall"    : round(r, 4),
            "f1"        : round(2 * p * r / max(p + r, 1e-6), 4),
            "AP50"      : round(float(metrics.box.ap50[arr_idx]), 4),
            "AP50-95"   : round(float(metrics.box.ap[arr_idx]),   4),
        })

    # Classes with no test-set instances (still useful to display as N/A)
    evaluated_ids = {int(c) for c in ap_class_index}
    missing = [(cid, name) for cid, name in metrics.names.items() if cid not in evaluated_ids]
    for cid, name in missing:
        class_rows.append({
            "id": cid, "class": name,
            "precision": None, "recall": None, "f1": None,
            "AP50": None, "AP50-95": None,
        })

    cls_df = pd.DataFrame(class_rows).set_index("id")
    cls_df = cls_df.sort_values("AP50", ascending=False, na_position="last")

    print("PER-CLASS METRICS (sorted by AP50)")
    print(f"Classes evaluated: {len(evaluated_ids)} / {len(metrics.names)}")
    print("=" * 70)
    print(cls_df.to_string())

    csv_path = OUTPUTS_DIR / "images" / "per_class_metrics.csv"
    cls_df.to_csv(csv_path)
    print(f"\nSaved -> {csv_path}")

In [ ]:
if WEIGHTS.exists():
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    colors = sns.color_palette("RdYlGn", len(cls_df))
    sorted_cls = cls_df.sort_values("AP50")

    for ax, metric, title in zip(axes,
                                  ["precision", "recall", "AP50"],
                                  ["Precision", "Recall", "AP@50"]):
        vals = sorted_cls[metric].values
        bars = ax.barh(sorted_cls["class"], vals,
                       color=sns.color_palette("RdYlGn", len(vals)))
        ax.set_xlim(0, 1.05)
        ax.set_xlabel(title)
        ax.set_title(f"Per-Class {title}")
        ax.axvline(0.70, color="navy", ls="--", alpha=0.5, label="0.70 target")
        for bar, v in zip(bars, vals):
            ax.text(min(v + 0.01, 1.0), bar.get_y() + bar.get_height()/2,
                    f"{v:.2f}", va="center", fontsize=9)
        ax.legend(fontsize=8)

    plt.suptitle("Per-Class Detection Metrics — Test Set", fontsize=14, fontweight="bold")
    plt.tight_layout()
    bar_path = OUTPUTS_DIR / "images" / "eval_per_class_bars.png"
    plt.savefig(bar_path, dpi=150)
    plt.show()
    print(f"Saved → {bar_path}")

## 5. Confusion Matrix

In [ ]:
def show_plot(src: Path, title: str, dest: Path, figsize=(12,9)):
    if not src.exists():
        print(f"Not found: {src}")
        return
    img = mpimg.imread(str(src))
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    plt.title(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(dest), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {dest}")

if WEIGHTS.exists():
    val_dir = Path(metrics.save_dir)
    show_plot(
        val_dir / "confusion_matrix.png",
        "Confusion Matrix — Test Set",
        OUTPUTS_DIR / "images" / "eval_confusion_matrix.png",
    )

**How to read the confusion matrix:**
- Each row = actual class, each column = predicted class.
- Diagonal = correct predictions.
- Off-diagonal = misclassifications.
- Background row = false negatives (missed detections).
- Background column = false positives (ghost detections).

## 6. Precision-Recall & F1 Curves

In [ ]:
if WEIGHTS.exists():
    show_plot(
        val_dir / "PR_curve.png",
        "Precision-Recall Curve — All Classes",
        OUTPUTS_DIR / "images" / "eval_pr_curve.png",
        figsize=(14, 8),
    )
    show_plot(
        val_dir / "F1_curve.png",
        "F1 Score vs Confidence Threshold",
        OUTPUTS_DIR / "images" / "eval_f1_curve.png",
        figsize=(14, 8),
    )

## 7. Visual Predictions on Test Images

Qualitative check: do predictions look correct on real images?

In [ ]:
if WEIGHTS.exists():
    from src.ppe_detection.utils import TEST_IMAGES
    random.seed(99)
    imgs = random.sample(list(TEST_IMAGES.glob("*.jpg")), 9)

    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    axes = axes.flatten()

    for ax, img_path in zip(axes, imgs):
        frame = cv2.imread(str(img_path))
        dets  = predict_image(model, frame, conf=0.35)
        ann   = draw_detections(frame, dets)
        ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{len(dets)} detections", fontsize=9)
        ax.axis("off")

    plt.suptitle("Model Predictions on Test Images (conf ≥ 0.35)", fontsize=14)
    plt.tight_layout()
    pred_path = OUTPUTS_DIR / "images" / "eval_predictions.png"
    plt.savefig(pred_path, dpi=150)
    plt.show()
    print(f"Saved → {pred_path}")

## 8. Weak Class Analysis

After reviewing metrics, fill in this table:

| Class | AP50 | Issue | Recommended Action |
|---|---|---|---|
| — | — | — | — |

**Common failure patterns:**
- Low recall → model misses objects (increase epochs, augmentation)
- Low precision → too many false positives (raise confidence threshold)
- `person_sin_casco` / `person_sin_chaleco` confusion → critical for safety — consider higher weight

## 9. Conclusions & Next Steps

**Deployment criteria:**
- mAP50 ≥ 0.70 overall
- Recall ≥ 0.75 for `person` (anchor for compliance logic)
- Recall ≥ 0.65 for violation classes (`person_sin_casco`, `person_sin_chaleco`)

**Next:** `05_image_inference.ipynb` — run inference with SAFE/UNSAFE compliance overlay.